# Dataset Interpretation – Cybersecurity Exercise Recommendation Project

## 1. Project Context
From the project description:
- **Goal:** Recommend cybersecurity exercises tailored to an organisation’s context, threats, and APT landscape.
- **Phase 1:** Build a baseline **item-based recommender** using **exercise metadata** (TTPs, threat groups, complexity, maturity, etc.) to find similar drills.
- **Datasets:**  
  - `orgs_full.csv` → Organisation profiles (**context**).  
  - `exercises_full.csv` → Exercise library (**content**).  
  - `ratings_train_full.csv` → Historical performance outcomes (**success metrics**).  

---

## 2. `orgs_full.csv` – Organisation Profiles
Each row = **one organisation** and its cyber posture.

| Column | Meaning in Project |
|--------|--------------------|
| **ORGID** | Unique org identifier; used to link with exercise history. |
| **Industry** | Sector or primary threat campaign type. |
| **Region** | Geographic location of the organisation. |
| **Size** | Organisation size (Small, Medium, Large). |
| **SecurityBudget** | Scale of investment (Low, Medium, High). |
| **PrimarySecurityTeam** | Security operation model (In-house, Outsourced, Hybrid). |
| **Maturity** | 1–5 score of the organisation’s overall **cybersecurity posture**. |
| **Complexity** | 1–5 score of IT/security environment complexity. |
| **ExerciseFrequency** | Number of cyber drills per year. |
| **Threats** | Types of threats most relevant to the organisation (semicolon-separated). |
| **TTPs** | MITRE ATT&CK techniques observed in their threat landscape. |
| **Aims** | High-level objectives (e.g., testing response to specific malware or APTs). |

**Interpretation:**  
- `Maturity` here = **org-wide cyber posture** (linked to security processes, not exercises).  
- Useful for clustering orgs based on **Threats, TTPs, and Aims** to discover archetypes (e.g., ransomware-prone orgs vs. insider-threat-prone).  

---

## 3. `exercises_full.csv` – Exercise Metadata
Each row = **one cyber training exercise**.

| Column | Meaning in Project |
|--------|--------------------|
| **EXID** | Unique exercise identifier. |
| **ExCreation** | Date/version of exercise creation. |
| **ExThreat** | Threat type simulated (Ransomware, Phishing, Trojan). |
| **ExTTPs** | MITRE ATT&CK techniques tested (semicolon-separated). |
| **ExCategories** | Higher-level tags (simulation, tabletop, malware, financial). |
| **ExGroups** | Associated adversary groups (Wizard Spider, APT29, FIN7). |
| **ExSoftware** | Tools/malware (Mimikatz, Cobalt Strike, TrickBot). |
| **ExStructure** | Structure (single-phase, multi-stage). |
| **ExMaturity** | 1–5 score of how realistic/advanced the scenario is. |
| **ExComplexity** | 1–5 score of difficulty/skill level required. |
| **ExLength** | Duration (minutes). |
| **ExAudience** | Targeted participants (SOC, management, hybrid). |
| **ExTradeCraftIntra** | Intra-technique skill richness. |
| **ExTradeCraftInter** | Cross-technique/multi-stage interconnectedness. |

**Interpretation:**  
- `ExMaturity` is different from org `Maturity`: here it represents **attack realism and impact**, closer to CVE severity.  
- Strongly tied to **TTP richness** and **tradecraft scores** (intra/inter).  

<div class="alert alert-block alert-warning">
    
**What “Intra-technique Tradecraft” is Intra = inside one tactic/technique.**
It measures how much detail and sophistication an exercise shows within a single MITRE ATT&CK technique or tactic.
**Example:**
Imagine an exercise simulating Credential Dumping.
- A low intra score (~0.1) might just show a simple mimic (e.g., “attacker runs mimikatz.exe once”).
- A high intra score (~0.8–1.0) would go deep:
    - multiple credential dumping sub-techniques (LSASS memory scrape, SAM database export, DCSync, etc.),
    - variations in tools (mimikatz, procdump, custom scripts),
    - evasion tricks (obfuscation, anti-AV bypass).

So, higher intra-tradecraft = the exercise digs deep into variants and tactics within a single path.

**Contrast with ExTradeCraftInter**

Inter = across multiple techniques/attack stages.
It measures how interconnected the exercise is — chaining different TTPs into a scenario.
**Example**
- A ransomware scenario might go:
- Phishing → Credential Access → Privilege Escalation → Exfiltration → Impact (encrypt).
That’s high inter-tradecraft (multi-stage, multi-technique).
</div>
---

## 4. `ratings_train_full.csv` – Performance Outcomes
Each row links an org with an exercise.

| Column | Meaning |
|--------|---------|
| **ORGID** | Organisation reference. |
| **EXID** | Exercise reference. |
| **ExerciseResults** | Numeric performance score. |
| **ExerciseRating** | Rating (1–5) of how well the org handled the exercise. |

**Interpretation:**  
- Provides the ground truth to **develop success metrics**.  
- Can evaluate whether org maturity correlates with better performance across exercises.  

---

## 5. How They Work Together
- **`orgs_full.csv`** = Context → what threats the org faces & what they want to prepare for.  
- **`exercises_full.csv`** = Content → library of available drills, defined by threat/TTPs/maturity.  
- **`ratings_train_full.csv`** = Outcomes → basis for metrics of success.  

**Phase 1 Workflow:**  
1. EDA:  
   - Compare org **Maturity (posture)** vs outcomes.  
   - Compare exercise **ExMaturity** vs TTP richness and tradecraft.  
   - Cluster orgs based on Threats/Groups/TTPs.  
2. Feature Engineering:  
   - Create a DTM from exercise metadata (TTPs, threats, groups, software, structure, audience).  
   - Add normalised numeric fields (`ExMaturity`, `ExComplexity`, `ExLength`, `ExTradeCraft*`).  
3. Similarity & Recommendation:  
   - Compute cosine similarity between exercises.  
   - Recommend new drills for an org, guided by similarity to past exercises.  
   - Explore tradecraft-based similarity metrics for richer recommendations.  


In [1]:
# Importing necesssary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the dataset
df_orgs = pd.read_csv("orgs_full.csv")
df_exs  = pd.read_csv("exercises_full.csv")

# Info and data types of the datasets & columns
df_orgs.info(), df_exs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ORGID                40 non-null     int64 
 1   Industry             40 non-null     object
 2   Region               40 non-null     object
 3   Size                 40 non-null     object
 4   SecurityBudget       40 non-null     object
 5   PrimarySecurityTeam  40 non-null     object
 6   Maturity             40 non-null     int64 
 7   Complexity           40 non-null     int64 
 8   ExerciseFrequency    40 non-null     int64 
 9   Threats              40 non-null     object
 10  TTPs                 40 non-null     object
 11  Aims                 40 non-null     object
dtypes: int64(4), object(8)
memory usage: 3.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  -----

(None, None)

In [2]:
df_orgs.head(), df_exs.head()

(   ORGID        Industry         Region    Size SecurityBudget  \
 0      1      Ransomware  South America   Small         Medium   
 1      2    Supply Chain         Europe   Large         Medium   
 2      3  Insider Threat  North America   Small         Medium   
 3      4  Banking Trojan         Africa  Medium            Low   
 4      5  Banking Trojan         Africa  Medium           High   
 
   PrimarySecurityTeam  Maturity  Complexity  ExerciseFrequency  \
 0            In-house         4           4                120   
 1            In-house         1           5                180   
 2          Outsourced         4           4                 90   
 3              Hybrid         1           4                300   
 4            In-house         4           5                180   
 
                                   Threats  \
 0                    Remote Access Trojan   
 1                          DDoS;Web Shell   
 2  Banking Trojan;Ransomware;Crypto Miner   
 3      

# Exploratory Data Analysis

1. Checking missing values
   